# 🏆 World Cup 2026 Predictor — Exploratory Data Analysis

Este notebook analiza los datasets del proyecto antes de entrenar el modelo:
1. Distribución de goles históricos
2. Rankings FIFA y valores de plantilla
3. Ratings Dixon-Coles (ataque/defensa)
4. Correlaciones entre features y goles
5. Análisis de penaltis por equipo
6. Forma reciente de los equipos del Mundial 2026

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Estilo
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

print('Librerías cargadas correctamente')

## 1. Carga de Datos

In [ ]:
results     = pd.read_csv(DATA_RAW / 'results.csv', parse_dates=['date'])
goalscorers = pd.read_csv(DATA_RAW / 'goalscorers.csv', parse_dates=['date'])
shootouts   = pd.read_csv(DATA_RAW / 'shootouts.csv',   parse_dates=['date'])

# Datos procesados (deben existir tras ejecutar --step prepare)
features    = pd.read_csv(DATA_PROCESSED / 'match_features.csv', parse_dates=['date'])
team_stats  = pd.read_csv(DATA_PROCESSED / 'team_stats.csv')
ratings     = pd.read_csv(DATA_PROCESSED / 'attack_defense_ratings.csv')

print(f'Partidos históricos : {len(results):,}')
print(f'Goleadores          : {len(goalscorers):,}')
print(f'Shootouts           : {len(shootouts):,}')
print(f'Features (train)    : {len(features):,} partidos x {features.shape[1]} columnas')
print(f'Equipos con ratings : {len(ratings)}')

## 2. Distribución de Goles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Goles por partido (local y visitante combinados)
all_goals = pd.concat([
    results['home_score'].rename('goals'),
    results['away_score'].rename('goals')
]).dropna().astype(int)

axes[0].hist(all_goals, bins=range(0, 15), edgecolor='white', color='steelblue', alpha=0.85)
axes[0].set_title('Distribución de Goles por Partido (histórico)')
axes[0].set_xlabel('Goles')
axes[0].set_ylabel('Frecuencia')

# Comparar con Poisson teórico
from scipy.stats import poisson
lam = all_goals.mean()
x = np.arange(0, 10)
expected = poisson.pmf(x, lam) * len(all_goals)
axes[0].plot(x + 0.5, expected, 'r--o', label=f'Poisson(λ={lam:.2f})', markersize=6)
axes[0].legend()

# Dataset de entrenamiento
axes[1].hist(features['home_goals'], bins=range(0, 12), alpha=0.7, label='Local', edgecolor='white')
axes[1].hist(features['away_goals'], bins=range(0, 12), alpha=0.7, label='Visitante', edgecolor='white')
axes[1].set_title('Goles en Dataset de Entrenamiento')
axes[1].set_xlabel('Goles')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

# Evolución de goles por año
results['year'] = results['date'].dt.year
yearly = results[results['year'] >= 1950].copy()
yearly['total_goals'] = yearly['home_score'] + yearly['away_score']
avg_by_year = yearly.groupby('year')['total_goals'].mean()
axes[2].plot(avg_by_year.index, avg_by_year.values, color='steelblue', linewidth=2)
axes[2].fill_between(avg_by_year.index, avg_by_year.values, alpha=0.15, color='steelblue')
axes[2].set_title('Media de Goles por Partido por Año')
axes[2].set_xlabel('Año')
axes[2].set_ylabel('Goles/partido')
axes[2].axhline(avg_by_year.mean(), color='red', linestyle='--', alpha=0.5, label='Media global')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/eda_goals_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Media goles/partido (histórico): {all_goals.mean():.3f}')
print(f'Media goles local (train):       {features["home_goals"].mean():.3f}')
print(f'Media goles visitante (train):   {features["away_goals"].mean():.3f}')

## 3. Rankings FIFA y Valores de Plantilla

In [ ]:
# Equipos del Mundial 2026
wc_teams = [
    'Spain','France','Germany','England','Argentina','Brazil',
    'Portugal','Netherlands','Morocco','Croatia','Colombia','Uruguay',
    'Mexico','United States','Canada','Japan','South Korea','Belgium',
    'Switzerland','Ecuador','Senegal','Norway','Egypt','Iran',
    'Australia','Sweden','Denmark','Poland','Algeria','Serbia',
    'Panama','Scotland'
]

# Buscar en team_stats los del Mundial
wc_stats = team_stats[team_stats['team'].isin(wc_teams)].copy()
wc_stats = wc_stats.dropna(subset=['fifa_rank']).sort_values('fifa_rank')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ranking FIFA
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(wc_stats)))
bars = axes[0].barh(wc_stats['team'], wc_stats['fifa_rank'], color=colors)
axes[0].invert_yaxis()
axes[0].set_title('Ranking FIFA — Equipos del Mundial 2026')
axes[0].set_xlabel('Posición FIFA (menor = mejor)')
axes[0].axvline(50, color='gray', linestyle='--', alpha=0.4, label='Rank 50')
axes[0].legend()

# Puntos FIFA vs forma reciente
sc = axes[1].scatter(
    wc_stats['fifa_points'], wc_stats['avg_goals_scored'],
    s=100, c=wc_stats['fifa_rank'], cmap='RdYlGn_r',
    alpha=0.8, edgecolors='white', linewidth=0.5
)
for _, row in wc_stats.iterrows():
    axes[1].annotate(row['team'][:3].upper(),
                     (row['fifa_points'], row['avg_goals_scored']),
                     fontsize=7, ha='center', va='bottom')
plt.colorbar(sc, ax=axes[1], label='Ranking FIFA')
axes[1].set_title('Puntos FIFA vs Goles Marcados (forma reciente)')
axes[1].set_xlabel('Puntos FIFA')
axes[1].set_ylabel('Media goles marcados (últimos 10 partidos)')

plt.tight_layout()
plt.savefig('../results/eda_fifa_rankings.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ratings Dixon-Coles — Ataque y Defensa

In [ ]:
wc_ratings = ratings[ratings['team'].isin(wc_teams)].copy()
wc_ratings = wc_ratings.sort_values('attack_rating', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 por ataque
top_atk = wc_ratings.head(20)
axes[0].barh(top_atk['team'], top_atk['attack_rating'],
             color=plt.cm.Greens(np.linspace(0.4, 0.9, len(top_atk))))
axes[0].invert_yaxis()
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Media global')
axes[0].set_title('Rating de Ataque — Dixon-Coles\n(>1 = por encima de la media)')
axes[0].set_xlabel('Attack Rating')
axes[0].legend()

# Top 20 por defensa (menor = mejor)
top_def = wc_ratings.sort_values('defense_rating').head(20)
axes[1].barh(top_def['team'], top_def['defense_rating'],
             color=plt.cm.Blues(np.linspace(0.9, 0.4, len(top_def))))
axes[1].invert_yaxis()
axes[1].axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Media global')
axes[1].set_title('Rating de Defensa — Dixon-Coles\n(<1 = mejor defensa)')
axes[1].set_xlabel('Defense Rating')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/eda_dixon_coles.png', dpi=150, bbox_inches='tight')
plt.show()

# Cuadrante ataque vs defensa
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(wc_ratings['attack_rating'], wc_ratings['defense_rating'],
           s=100, alpha=0.7, color='steelblue', edgecolors='white')
for _, row in wc_ratings.iterrows():
    ax.annotate(row['team'], (row['attack_rating'], row['defense_rating']),
                fontsize=8, ha='center', va='bottom')
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.4)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Attack Rating (→ mejor ataque)')
ax.set_ylabel('Defense Rating (↓ mejor defensa)')
ax.set_title('Cuadrante Ataque vs Defensa — Equipos del Mundial 2026')

# Etiquetas de cuadrantes
ax.text(ax.get_xlim()[1]*0.98, ax.get_ylim()[0]*1.02, 'Buen ataque\nBuena defensa',
        ha='right', va='bottom', fontsize=9, color='green', alpha=0.6)
ax.text(ax.get_xlim()[0]*1.02, ax.get_ylim()[1]*0.98, 'Mal ataque\nMala defensa',
        ha='left', va='top', fontsize=9, color='red', alpha=0.6)

plt.tight_layout()
plt.savefig('../results/eda_attack_defense_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Valores de Plantilla (Transfermarkt)

In [ ]:
squad_path = DATA_PROCESSED / 'squad_features.csv'
if squad_path.exists():
    squad = pd.read_csv(squad_path)
    wc_squad = squad[squad['team'].isin(wc_teams)].sort_values('squad_value_M', ascending=False).head(25)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # Valor de plantilla
    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(wc_squad)))
    axes[0].barh(wc_squad['team'], wc_squad['squad_value_M'], color=colors[::-1])
    axes[0].invert_yaxis()
    axes[0].set_title('Valor de Plantilla — Top 25 Equipos del Mundial 2026')
    axes[0].set_xlabel('Valor Total en M€')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M€'))

    # Valor plantilla vs form score
    axes[1].scatter(wc_squad['squad_value_M'], wc_squad['squad_form_score'],
                    s=100, alpha=0.8, color='steelblue', edgecolors='white')
    for _, row in wc_squad.iterrows():
        axes[1].annotate(row['team'][:3].upper(),
                         (row['squad_value_M'], row['squad_form_score']),
                         fontsize=7, ha='center', va='bottom')
    axes[1].set_title('Valor de Plantilla vs Forma Reciente')
    axes[1].set_xlabel('Valor Plantilla (M€)')
    axes[1].set_ylabel('Squad Form Score (Transfermarkt)')

    plt.tight_layout()
    plt.savefig('../results/eda_squad_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('squad_features.csv no encontrado. Ejecuta primero: python main.py --step prepare')

## 6. Correlaciones entre Features y Goles

In [ ]:
num_cols = [
    'fifa_points_home','fifa_rank_home','avg_scored_home','avg_conceded_home',
    'win_rate_home','attack_rating_home','defense_rating_home',
    'squad_value_home','squad_form_home','value_ratio','home_goals'
]
available = [c for c in num_cols if c in features.columns]
corr_df = features[available].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación — Features del Modelo vs Goles Locales')
plt.tight_layout()
plt.savefig('../results/eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlaciones con home_goals
if 'home_goals' in available:
    corrs = corr_df['home_goals'].drop('home_goals').sort_values(key=abs, ascending=False)
    print('\nCorrelaciones con goles del equipo local:')
    for feat, val in corrs.items():
        bar = '█' * int(abs(val) * 20)
        sign = '+' if val > 0 else '-'
        print(f'  {feat:<30} {sign}{abs(val):.3f}  {bar}')

## 7. Historial de Penaltis — Equipos del Mundial 2026

In [ ]:
# Calcular tasas con decaimiento temporal (igual que el modelo)
import numpy as np

df_sh = pd.read_csv(DATA_RAW / 'shootouts.csv', parse_dates=['date'])
reference_date = df_sh['date'].max()
DECAY_WEEKS = 52 * 10

records = {}
for _, row in df_sh.iterrows():
    weeks_ago = max((reference_date - row['date']).days / 7, 0)
    weight = np.exp(-np.log(2) * weeks_ago / DECAY_WEEKS)
    for team in [row['home_team'], row['away_team']]:
        if team not in records:
            records[team] = {'won_w': 0.0, 'total_w': 0.0, 'total': 0}
        records[team]['total_w'] += weight
        records[team]['total'] += 1
        if team == row['winner']:
            records[team]['won_w'] += weight

penalty_rates = {}
for team, r in records.items():
    smoothed = (r['won_w'] + 3 * 0.5) / (r['total_w'] + 3)
    penalty_rates[team] = {'rate': smoothed, 'total': r['total']}

# Filtrar equipos del Mundial con al menos 2 tandas
wc_penalties = {t: v for t, v in penalty_rates.items()
                if t in wc_teams and v['total'] >= 2}
pen_df = pd.DataFrame([
    {'team': t, 'rate': v['rate'], 'total': v['total']}
    for t, v in wc_penalties.items()
]).sort_values('rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn(pen_df['rate'])
bars = ax.barh(pen_df['team'], pen_df['rate'], color=colors)
ax.axvline(0.5, color='black', linestyle='--', alpha=0.5, label='50% (base)')
for bar, (_, row) in zip(bars, pen_df.iterrows()):
    ax.text(row['rate'] + 0.005, bar.get_y() + bar.get_height()/2,
            f"{row['rate']:.2f} ({int(row['total'])} tandas)",
            va='center', fontsize=8)
ax.set_xlim(0, 0.9)
ax.set_title('Tasa de Victoria en Penaltis — Equipos del Mundial 2026\n(con decaimiento temporal 10 años + suavizado bayesiano)')
ax.set_xlabel('Probabilidad de ganar la tanda')
ax.legend()
plt.tight_layout()
plt.savefig('../results/eda_penalty_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Goleadores en Mundiales 2018 y 2022

In [ ]:
wc_results = results[results['tournament'] == 'FIFA World Cup'].copy()
wc_2018_dates = set(wc_results[wc_results['date'].dt.year == 2018]['date'].astype(str))
wc_2022_dates = set(wc_results[wc_results['date'].dt.year == 2022]['date'].astype(str))

gs = goalscorers.copy()
gs['date_str'] = gs['date'].astype(str)
gs = gs[gs['own_goal'] == False]

gs_2018 = gs[gs['date_str'].isin(wc_2018_dates)]
gs_2022 = gs[gs['date_str'].isin(wc_2022_dates)]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, gs_wc, year in [(axes[0], gs_2018, 2018), (axes[1], gs_2022, 2022)]:
    top = gs_wc.groupby(['scorer', 'team'])['scorer'].count().nlargest(12).reset_index(name='goals')
    colors = plt.cm.YlOrRd(np.linspace(0.4, 0.9, len(top)))
    ax.barh(top['scorer'] + '\n(' + top['team'] + ')', top['goals'], color=colors[::-1])
    ax.invert_yaxis()
    ax.set_title(f'Top Goleadores — Mundial {year}')
    ax.set_xlabel('Goles')

plt.tight_layout()
plt.savefig('../results/eda_world_cup_scorers.png', dpi=150, bbox_inches='tight')
plt.show()

# Goles por equipo en ambos mundiales
team_goals_2018 = gs_2018.groupby('team')['scorer'].count().rename('2018')
team_goals_2022 = gs_2022.groupby('team')['scorer'].count().rename('2022')
combined = pd.concat([team_goals_2018, team_goals_2022], axis=1).fillna(0)
combined['total'] = combined['2018'] + combined['2022']
combined = combined[combined.index.isin(wc_teams)].sort_values('total', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(combined))
ax.bar(x - 0.2, combined['2018'], 0.35, label='2018', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, combined['2022'], 0.35, label='2022', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(combined.index, rotation=45, ha='right')
ax.set_title('Goles por Equipo en Mundiales 2018 y 2022')
ax.set_ylabel('Goles')
ax.legend()
plt.tight_layout()
plt.savefig('../results/eda_team_wc_goals.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Jugadores del Mundial 2026 — Valores de Mercado

In [ ]:
player_path = DATA_PROCESSED / 'player_dataset.csv'
if player_path.exists():
    players = pd.read_csv(player_path)
    wc_players = players[players['country_of_citizenship'].isin(wc_teams)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Top 20 jugadores por valor de mercado
    top_players = wc_players.nlargest(20, 'market_value_in_eur')
    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(top_players)))
    axes[0].barh(
        top_players['name'] + ' (' + top_players['country_of_citizenship'].str[:3] + ')',
        top_players['market_value_in_eur'] / 1e6,
        color=colors[::-1]
    )
    axes[0].invert_yaxis()
    axes[0].set_title('Top 20 Jugadores por Valor de Mercado')
    axes[0].set_xlabel('Valor de Mercado (M€)')

    # Distribución de valores por posición
    pos_vals = wc_players[wc_players['market_value_in_eur'] > 0].groupby('position')['market_value_in_eur'].apply(list)
    pos_order = ['Goalkeeper', 'Defender', 'Midfield', 'Attack']
    data_plot = [pos_vals.get(p, [0]) for p in pos_order]
    bp = axes[1].boxplot([np.array(d)/1e6 for d in data_plot],
                         labels=pos_order, patch_artist=True,
                         medianprops=dict(color='red', linewidth=2))
    colors_box = ['#a8d8ea', '#aa96da', '#fcbad3', '#ffffd2']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    axes[1].set_title('Distribución de Valor de Mercado por Posición')
    axes[1].set_ylabel('Valor (M€)')
    axes[1].set_ylim(0, 220)

    plt.tight_layout()
    plt.savefig('../results/eda_player_values.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Total jugadores activos en equipos del Mundial: {len(wc_players):,}')
    print(f'Valor medio por jugador: {wc_players["market_value_in_eur"].mean()/1e6:.1f}M€')
    print(f'Jugador más valioso: {wc_players.nlargest(1, "market_value_in_eur").iloc[0]["name"]}')
else:
    print('player_dataset.csv no encontrado. Ejecuta primero: python main.py --step prepare')

## 10. Resumen del Dataset de Entrenamiento

In [ ]:
print('=== RESUMEN DEL DATASET DE ENTRENAMIENTO ===')
print(f'Partidos: {len(features):,}')
print(f'Features: {features.shape[1]}')
print(f'Rango:    {features["date"].min().date()} — {features["date"].max().date()}')
print()

print('Distribución por torneo:')
for t, n in features['tournament'].value_counts().head(10).items():
    pct = n / len(features) * 100
    bar = '█' * int(pct / 2)
    print(f'  {t:<40} {n:>5} ({pct:.1f}%)  {bar}')

print()
print('Estadísticas de los targets (goles):')
print(features[['home_goals', 'away_goals']].describe().round(3).to_string())

print()
print('Features disponibles:')
numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
for i, col in enumerate(numeric_cols):
    if col not in ['home_goals', 'away_goals']:
        print(f'  {i+1:>2}. {col}')